# 🛡️ ChainSleuth — Automated Forensic ML Training Pipeline

End-to-end automated training pipeline for **ChainSleuth** (GraphSAGE GNN, XGBoost typologies, Isolation Forest).

- Auto-unzips all files
- Auto-locates all datasets recursively (zero hardcoded path errors)
- Packages all trained models into `chainsleuth_models.zip` for instant download

### 1. Upload Kaggle API Key (`kaggle.json`)

In [ ]:
import os
import shutil
from google.colab import files

print("Upload your kaggle.json token file:")
uploaded = files.upload()

os.makedirs('/root/.config/kaggle', exist_ok=True)
for fn in uploaded.keys():
    if fn.endswith('kaggle.json'):
        shutil.move(fn, '/root/.config/kaggle/kaggle.json')
        os.chmod('/root/.config/kaggle/kaggle.json', 0o600)
        print("\n✅ Kaggle API configured successfully!")
        break
else:
    if os.path.exists('/root/.config/kaggle/kaggle.json'):
        print("\n✅ Existing Kaggle config found.")
    else:
        print("\n⚠️ No kaggle.json uploaded. Please ensure kaggle.json is uploaded.")

### 2. Install Dependencies

In [ ]:
!pip install -q torch-geometric xgboost scikit-learn onnx onnxruntime joblib pandas numpy networkx
print("✅ Libraries installed!")

### 3. Download & Auto-Extract All Datasets

In [ ]:
import os
import glob
import zipfile

# Create base directories
os.makedirs('/content/data/elliptic', exist_ok=True)
os.makedirs('/content/data/eth_phishing', exist_ok=True)
os.makedirs('/content/data/babd13', exist_ok=True)
os.makedirs('/content/data/eth_activity', exist_ok=True)
os.makedirs('/content/data/ofac', exist_ok=True)
os.makedirs('/content/models', exist_ok=True)

print("1/5 Pulling Elliptic Bitcoin dataset...")
!kaggle datasets download ellipticco/elliptic-data-set -p /content/data/elliptic --unzip

print("\n2/5 Pulling Ethereum Phishing Network dataset...")
!kaggle datasets download xblock/ethereum-phishing-transaction-network -p /content/data/eth_phishing --unzip

print("\n3/5 Pulling BABD-13 Bitcoin behavior dataset...")
!kaggle datasets download lemonx/babd13 -p /content/data/babd13 --unzip

print("\n4/5 Pulling Ethereum Fraud dataset...")
!kaggle datasets download vagifa/ethereum-fraud-detection-dataset -p /content/data/eth_activity --unzip

print("\n5/5 Pulling OFAC Sanctions list...")
!wget -q https://www.treasury.gov/ofac/downloads/sdn.csv -O /content/data/ofac/sdn.csv

# Ensure ALL downloaded zip files are unzipped everywhere under /content/data
for zpath in glob.glob('/content/data/**/*.zip', recursive=True):
    target_dir = os.path.dirname(zpath)
    try:
        with zipfile.ZipFile(zpath, 'r') as zip_ref:
            zip_ref.extractall(target_dir)
        print(f"Extracted: {zpath}")
    except Exception as e:
        print(f"Warning unpacking {zpath}: {e}")

# Delete raw parquet blocks > 50MB to keep disk usage low
for f in glob.glob('/content/data/**/*.parquet', recursive=True):
    if os.path.getsize(f) > 50 * 1024 * 1024:
        os.remove(f)

print("\n✅ All datasets downloaded and extracted successfully!")

### 4. Train Component 1: GraphSAGE GNN (Auto-locating data)

In [ ]:
import glob
import os
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from sklearn.metrics import classification_report, f1_score

# Dynamic path locator
def locate_file(filename):
    hits = glob.glob(f"/content/**/{filename}", recursive=True)
    if not hits:
        raise FileNotFoundError(f"Could not locate {filename} under /content")
    return hits[0]

feat_path = locate_file('elliptic_txs_features.csv')
edge_path = locate_file('elliptic_txs_edgelist.csv')
cls_path  = locate_file('elliptic_txs_classes.csv')

print(f"Loading Elliptic graph:\n - {feat_path}\n - {edge_path}\n - {cls_path}")
features_df = pd.read_csv(feat_path, header=None)
edges_df = pd.read_csv(edge_path)
classes_df = pd.read_csv(cls_path)

# Map node IDs
tx_ids = features_df.iloc[:, 0].tolist()
id_map = {tx_id: idx for idx, tx_id in enumerate(tx_ids)}

# Node features: 50 normalized behavioral features
x = torch.tensor(features_df.iloc[:, 1:51].values, dtype=torch.float)

valid_edges = edges_df[edges_df['txId1'].isin(id_map) & edges_df['txId2'].isin(id_map)]
edge_src = [id_map[tx] for tx in valid_edges['txId1']]
edge_dst = [id_map[tx] for tx in valid_edges['txId2']]
edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)

# 1 = Illicit (Criminal), 2 = Licit (Clean), unknown = -1
label_dict = classes_df.set_index('txId')['class'].to_dict()
y_list = [1 if str(label_dict.get(tx, '')) == '1' else (0 if str(label_dict.get(tx, '')) == '2' else -1) for tx in tx_ids]
y = torch.tensor(y_list, dtype=torch.long)

graph_data = Data(x=x, edge_index=edge_index, y=y)
print(f"\nGraph built: {graph_data.num_nodes:,} nodes, {graph_data.num_edges:,} edges")

# Train/Val/Test splits
labeled_idx = (graph_data.y != -1).nonzero(as_tuple=True)[0]
perm = torch.randperm(len(labeled_idx))
n_train = int(0.70 * len(labeled_idx))
n_val = int(0.15 * len(labeled_idx))

train_idx = labeled_idx[perm[:n_train]]
val_idx = labeled_idx[perm[n_train:n_train+n_val]]
test_idx = labeled_idx[perm[n_train+n_val:]]

num_illicit = (graph_data.y[train_idx] == 1).sum().item()
num_licit = (graph_data.y[train_idx] == 0).sum().item()
pos_weight = torch.tensor([1.0, float(num_licit / max(1, num_illicit))])

class GraphSAGE(torch.nn.Module):
    def __init__(self, in_dim=50, hidden_dim=64, out_dim=2):
        super().__init__()
        self.conv1 = SAGEConv(in_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, 32)
        self.fc = torch.nn.Linear(32, out_dim)
        self.dropout = torch.nn.Dropout(0.3)

    def forward(self, x, edge_index):
        h = self.conv1(x, edge_index)
        h = F.relu(h)
        h = self.dropout(h)
        h = self.conv2(h, edge_index)
        h = F.relu(h)
        return self.fc(h)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on {device}...")
model = GraphSAGE(in_dim=50).to(device)
x_dev, edge_dev, y_dev = graph_data.x.to(device), graph_data.edge_index.to(device), graph_data.y.to(device)
weight_dev = pos_weight.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)

for epoch in range(1, 101):
    model.train()
    optimizer.zero_grad()
    out = model(x_dev, edge_dev)
    loss = F.cross_entropy(out[train_idx], y_dev[train_idx], weight=weight_dev)
    loss.backward()
    optimizer.step()

    if epoch % 25 == 0 or epoch == 100:
        model.eval()
        with torch.no_grad():
            val_preds = out[val_idx].argmax(dim=1)
            val_f1 = f1_score(y_dev[val_idx].cpu(), val_preds.cpu(), pos_label=1)
            print(f"Epoch {epoch:03d} | Train Loss: {loss.item():.4f} | Val F1: {val_f1:.4f}")

# Export ONNX
model.eval().to('cpu')
dummy_x = torch.randn(4, 50, dtype=torch.float)
dummy_edges = torch.tensor([[0, 1, 2], [1, 2, 3]], dtype=torch.long)
torch.onnx.export(
    model, (dummy_x, dummy_edges), '/content/models/graphsage_risk.onnx',
    input_names=['x', 'edge_index'], output_names=['logits'],
    dynamic_axes={'x': {0: 'num_nodes'}, 'edge_index': {1: 'num_edges'}, 'logits': {0: 'num_nodes'}},
    opset_version=14
)
torch.save(model.state_dict(), '/content/models/graphsage_risk.pt')
print("\n✅ GraphSAGE model trained & exported: /content/models/graphsage_risk.onnx")

### 5. Train Component 2: XGBoost Multi-Class Typology Classifiers

In [ ]:
import pandas as pd
import numpy as np
import joblib
import glob
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Locate Ethereum transaction dataset
eth_candidates = glob.glob('/content/data/**/transaction*.csv', recursive=True)
if eth_candidates:
    eth_df = pd.read_csv(eth_candidates[0]).dropna()
    eth_df.columns = [c.strip() for c in eth_df.columns]
    sent_tx = eth_df.get('Sent tnx', pd.Series(np.zeros(len(eth_df))))
    rec_tx = eth_df.get('Received Tnx', pd.Series(np.zeros(len(eth_df))))
    uniq_sent = eth_df.get('Unique Sent To Addresses', pd.Series(np.zeros(len(eth_df))))
    uniq_rec = eth_df.get('Unique Received From Addresses', pd.Series(np.zeros(len(eth_df))))
    val_sent = eth_df.get('total ether sent', pd.Series(np.zeros(len(eth_df))))
    val_rec = eth_df.get('total ether received', pd.Series(np.zeros(len(eth_df))))
    avg_interval = eth_df.get('Avg min between sent tnx', pd.Series(np.zeros(len(eth_df))))
    time_diff = eth_df.get('Time Diff between first and last (Mins)', pd.Series(np.zeros(len(eth_df))))
    flag = eth_df.get('FLAG', pd.Series(np.zeros(len(eth_df)))).astype(int)
else:
    # Fallback to Elliptic features
    sent_tx = features_df.iloc[:, 2]
    rec_tx = features_df.iloc[:, 1]
    uniq_sent = features_df.iloc[:, 4]
    uniq_rec = features_df.iloc[:, 3]
    val_sent = features_df.iloc[:, 6]
    val_rec = features_df.iloc[:, 5]
    avg_interval = features_df.iloc[:, 7]
    time_diff = features_df.iloc[:, 8]
    flag = (classes_df['class'] == '1').astype(int)

X = pd.DataFrame({
    'in_degree': rec_tx,
    'out_degree': sent_tx,
    'fan_in_count': uniq_rec,
    'fan_out_count': uniq_sent,
    'in_volume': val_rec,
    'out_volume': val_sent,
    'pass_through_ratio': (val_sent / (val_rec + 1e-9)).clip(0, 5),
    'avg_interval_mins': avg_interval,
    'active_duration_mins': time_diff,
    'total_transacted': val_rec + val_sent
})

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
joblib.dump(scaler, '/content/models/typology_scaler.pkl')

# Typologies
y_peeling = ((flag == 1) & (X['pass_through_ratio'].between(0.7, 1.25)) & (X['out_degree'] >= 2)).astype(int)
y_fanout = ((X['fan_out_count'] >= 5) & (X['fan_out_count'] > X['fan_in_count'])).astype(int)
y_mixer = ((flag == 1) & (X['fan_out_count'] >= 3) & (X['pass_through_ratio'] > 0.95)).astype(int)
y_burner = ((X['active_duration_mins'] < 120) & (X['out_degree'] >= 2) & (X['pass_through_ratio'] >= 0.9)).astype(int)

typology_targets = {
    'peeling_chain': y_peeling,
    'fan_out': y_fanout,
    'coinjoin_mixer': y_mixer,
    'burner_wallet': y_burner,
}

for name, y_target in typology_targets.items():
    pos = y_target.sum()
    neg = len(y_target) - pos
    scale_weight = float(neg / max(1, pos))
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_target, test_size=0.2, random_state=42, stratify=y_target)
    clf = XGBClassifier(n_estimators=150, max_depth=5, learning_rate=0.08, scale_pos_weight=scale_weight, eval_metric='logloss')
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)
    print(f"\n--- Model: {name} (Positive samples: {pos}) ---")
    print(classification_report(y_test, preds, zero_division=0))
    joblib.dump(clf, f'/content/models/{name}_xgb.pkl')
    print(f"✅ Saved /content/models/{name}_xgb.pkl")

print("\n✅ All XGBoost typology models trained & saved!")

### 6. Train Component 3: Isolation Forest Anomaly Detector

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

print("Training Isolation Forest on clean baseline transactions...")
clean_mask = (flag == 0)
X_clean = X[clean_mask]

iso_scaler = StandardScaler()
X_clean_scaled = iso_scaler.fit_transform(X_clean)
joblib.dump(iso_scaler, '/content/models/anomaly_scaler.pkl')

iso_forest = IsolationForest(n_estimators=200, contamination=0.05, random_state=42, n_jobs=-1)
iso_forest.fit(X_clean_scaled)

joblib.dump(iso_forest, '/content/models/isolation_forest.pkl')
print("✅ Saved /content/models/isolation_forest.pkl and anomaly_scaler.pkl")

### 7. Package & Download Models (`chainsleuth_models.zip`)

In [ ]:
import shutil
from google.colab import files

print("Generated model artifacts in /content/models/:")
!ls -lh /content/models

zip_filename = "/content/chainsleuth_models"
shutil.make_archive(zip_filename, 'zip', "/content/models")

print(f"\n📦 Models packaged to {zip_filename}.zip")
print("Starting download to your browser...")
files.download(f"{zip_filename}.zip")
print("\n👉 Once downloaded, unzip the contents into: backend/models/")